# 免費 GPU 四合一：出圖・短片・對話・放大

Kaggle 免費 T4 x2 專用 notebook，一個介面用齊四個模型：

| Tab | 模型 | 用途 | 大細 |
| --- | --- | --- | --- |
| 出圖 | Qwen-Image-2.1（FP8，unsloth 量化版） | 文字出圖＋改圖 | 約 17 GB |
| 短片 | Wan 2.2 TI2V-5B | 文字／圖片變短片 | 約 30 GB |
| 對話 | Qwen3-8B（4-bit，bitsandbytes） | 本地大模型傾偈 | 約 15 GB 下載，4-bit 佔 5 GB 顯存 |
| 放大 | Real-ESRGAN x4plus | 舊相放大修復到 4x | 64 MB |

**用法**：由上而下跑一次，最後一格會開 Gradio 介面，俾一條公開連結，喺電話／電腦瀏覽器開就用得。

**要知嘅限制**（唔好誤會係長開 server）：
- Kaggle session 最長 12 小時，閒置 20 分鐘會斷線；每週 GPU 約 30 小時。
- 每次開新 session 都要重新下載模型（第一次約 20–40 分鐘，視乎網速），因為 Kaggle 唔會幫你留住幾十 GB 權重。
- 模型係逐個按需要先載入（撳邊個 tab 先 load 邊個），唔會一次過食晒兩張 T4 嘅顯存。
- Qwen-Image-2.1 係 Qwen Research License（非商業用），個人玩冇問題。
- 出短片最慢：480p、2 秒片喺 T4 上都要幾分鐘，720p 會再慢幾倍，唔好心急。
- 版本全部釘死（diffusers 0.41.0＋transformers 5.17.0，Qwen 官方文件指定嘅配對；對話唔用 AWQ，改用 bitsandbytes 4-bit，唔使由源碼 build 任何嘢；放大嗰個唔使裝 basicsr，RRDBNet 程式碼直接寫喺 notebook 入面）。
- 安裝分開幾行跑：其中一個套件裝唔到都唔會拖垮其他；煙霧測試亦各自包住，個別失敗唔會令成個 notebook 停低，Gradio 照開。
- 注意：Kaggle import 嗰陣會自動跑一轉 CPU 版（未掛 GPU 嗰陣），嗰轉嘅結果唔準，唔好理佢；要睇 draft session（GPU 嗰轉）先係真。

In [ ]:
# 1) 環境檢查
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), '| GPU 數量:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} ({p.total_memory / 2**30:.1f} GB)')

In [ ]:
# 2) 安裝套件（每行獨立 pip，邊行失敗都唔會拖垮其他行；版本釘死，唔好亂升）
# diffusers 0.41.0 + transformers 5.17.0 係 Qwen 官方文件指定嘅配對（0.41.0 係第一個有 Qwen-Image 2.1 嘅正式版）
# 唔裝 torchao（最新版會將 torch 由 2.11 升到 2.15）、唔裝 autoawq／gptqmodel（gptqmodel 喺呢個環境冇預製 wheel，要由源碼 build CUDA，成兩個鐘都未完；對話改用 bitsandbytes，全部係預製 wheel）
# 唔裝 basicsr／realesrgan：喺 Kaggle 嘅 Python 3.13 裝唔到，放大改用第 7 格自帶嘅 RRDBNet 程式碼
%pip install -q hf_transfer
%pip install -q "diffusers==0.41.0" "transformers==5.17.0" accelerate
%pip install -q bitsandbytes
%pip install -q gradio imageio imageio-ffmpeg opencv-python-headless
import importlib
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '（應該係 2.11.0+cu130，如果變咗 2.15 即係有嘢偷偷升咗 torch）')
for _m in ['diffusers', 'transformers', 'accelerate', 'gradio', 'bitsandbytes']:
    try:
        _mod = importlib.import_module(_m)
        print(f'{_m}: OK', getattr(_mod, '__version__', ''))
    except Exception as _e:
        print(f'{_m}: FAILED - {_e}')
print('套件安裝完成')

In [ ]:
# 3) 設定 + 共用工具
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'  # 用 hf_transfer 加快下載
import gc
import torch

_LOADED = {}

def unload_all():
    """卸載全部已載入嘅模型，釋放顯存（換模型之前用）。"""
    for name in list(_LOADED.keys()):
        print(f'卸載 {name} ...')
        del _LOADED[name]
    gc.collect()
    torch.cuda.empty_cache()
    print('顯存已釋放')

print('設定完成。')

In [ ]:
# 4) 出圖：Qwen-Image-2.1（FP8 量化版，unsloth）
# 全精度 BF16 成 33GB 放唔入 T4，呢個 FP8 版得 17GB 左右，畫質接近。
# 8 步 Turbo 版更快，日後想快啲可以換 unsloth/Qwen-Image-2.1-Turbo-FP8 嘅 transformer。
def load_qwen_image():
    if 'image' in _LOADED:
        return _LOADED['image']
    unload_all()
    from diffusers import QwenImage21Pipeline
    print('載入 Qwen-Image-2.1 (FP8)... 第一次下載約 17GB，要等一陣')
    try:
        pipe = QwenImage21Pipeline.from_pretrained(
            'unsloth/Qwen-Image-2.1-FP8',
            torch_dtype=torch.float8_e4m3fn,
            device_map='balanced',
        )
    except Exception as e:
        print('FP8 載入失敗，轉用官方 BF16 + CPU offload（會慢啲）:', e)
        pipe = QwenImage21Pipeline.from_pretrained('Qwen/Qwen-Image-2.1', torch_dtype=torch.bfloat16)
        pipe.enable_model_cpu_offload()
    _LOADED['image'] = pipe
    print('Qwen-Image-2.1 已載入')
    return pipe

def make_image(prompt, width=1024, height=1024, steps=25, seed=0):
    pipe = load_qwen_image()
    gen = torch.Generator(device='cuda').manual_seed(seed) if seed else None
    img = pipe(prompt=prompt, width=int(width), height=int(height),
               num_inference_steps=int(steps), generator=gen).images[0]
    return img

In [ ]:
# 5) 短片：Wan 2.2 TI2V-5B（文字出片／圖片變片）
# 官方寫明 24GB VRAM 先跑得順，T4 得 16GB，所以用 CPU offload + 480p，慢但跑得郁。
def load_wan():
    if 'video' in _LOADED:
        return _LOADED['video']
    unload_all()
    from diffusers import WanPipeline, AutoencoderKLWan
    model_id = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
    print('載入 Wan 2.2 TI2V-5B... 第一次下載約 30GB，要等一陣')
    vae = AutoencoderKLWan.from_pretrained(model_id, subfolder='vae', torch_dtype=torch.float32)
    pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
    pipe.enable_model_cpu_offload()  # T4 16GB 必需，否則 OOM
    _LOADED['video'] = pipe
    print('Wan 2.2 已載入')
    return pipe

WAN_NEG = 'low quality, blurry, static, deformed, ugly, bad hands, bad face, watermark, text, subtitle'

def make_video(prompt, image=None, width=832, height=480, num_frames=33, steps=25):
    pipe = load_wan()
    from diffusers.utils import export_to_video
    out = pipe(prompt=prompt, negative_prompt=WAN_NEG, image=image,
               height=int(height), width=int(width), num_frames=int(num_frames),
               guidance_scale=5.0, num_inference_steps=int(steps)).frames[0]
    path = '/kaggle/working/wan_output.mp4'
    export_to_video(out, path, fps=16)
    return path

In [ ]:
# 6) 對話：Qwen3-8B，用 bitsandbytes 4-bit（NF4）量化
# 唔用 AWQ 版：gptqmodel 喺呢個環境冇預製 wheel，要由源碼 build CUDA，v3 嗰轉 build 咗兩個鐘都未完。
# bitsandbytes 全部係預製 wheel，裝幾秒就完；模型本身係同一個 Qwen3-8B（全精度下載約 15GB，4-bit 之後顯存佔約 5GB）。
def load_qwen3():
    if 'chat' in _LOADED:
        return _LOADED['chat']
    unload_all()
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    print('載入 Qwen3-8B (4-bit)... 第一次下載約 15GB，要等一陣')
    tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-8B')
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                               bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(
        'Qwen/Qwen3-8B', quantization_config=quant,
        torch_dtype=torch.float16, device_map='auto')
    _LOADED['chat'] = (model, tok)
    print('Qwen3-8B 已載入')
    return model, tok

def chat_reply(user_text, max_tokens=512):
    model, tok = load_qwen3()
    messages = [{'role': 'user', 'content': user_text}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok([text], return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=int(max_tokens),
                         do_sample=True, temperature=0.7, top_p=0.9)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# 7) 放大：Real-ESRGAN x4plus（權重 64MB）
# basicsr／realesrgan 呢兩個套件喺 Kaggle 嘅 Python 3.13 係裝唔到嘅（setup.py 失敗），所以 RRDBNet 架構直接寫喺呢度，唔使靠佢哋；
# 權重檔本身（RealESRGAN_x4plus.pth）同 basicsr 嗰個完全一樣，直接用 torch.load 讀，唔使經 realesrgan 套件。
import torch.nn as nn
import torch.nn.functional as F

class ResidualDenseBlock_5C(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.conv1 = nn.Conv2d(num_feat, num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat + num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4 * num_grow_ch, num_feat, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, num_feat, num_grow_ch=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock_5C(num_feat, num_grow_ch)
        self.rdb2 = ResidualDenseBlock_5C(num_feat, num_grow_ch)
        self.rdb3 = ResidualDenseBlock_5C(num_feat, num_grow_ch)
    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out * 0.2 + x

class RRDBNet(nn.Module):
    def __init__(self, num_in, num_out, num_feat, num_block, num_grow_ch, scale=4):
        super().__init__()
        self.conv_first = nn.Conv2d(num_in, num_feat, 3, 1, 1)
        self.body = nn.ModuleList([RRDB(num_feat, num_grow_ch) for _ in range(num_block)])
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = feat
        for block in self.body:
            body_feat = block(body_feat)
        body_feat = self.conv_body(body_feat)
        feat = feat + body_feat
        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode='nearest')))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode='nearest')))
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return out

def load_esrgan():
    if 'upscale' in _LOADED:
        return _LOADED['upscale']
    import os
    import urllib.request
    weight_url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
    weight_path = '/kaggle/working/weights/RealESRGAN_x4plus.pth'
    if not os.path.exists(weight_path):
        os.makedirs('/kaggle/working/weights', exist_ok=True)
        print('下載 Real-ESRGAN x4plus 權重 (64MB)...')
        urllib.request.urlretrieve(weight_url, weight_path)
    model = RRDBNet(num_in=3, num_out=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    ckpt = torch.load(weight_path, map_location='cpu', weights_only=True)
    state = ckpt.get('params-ema', ckpt.get('params', ckpt)) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state, strict=True)
    model.eval().cuda()
    _LOADED['upscale'] = model
    print('Real-ESRGAN 已載入')
    return model

def upscale_image(pil_image, tile=512, tile_pad=16):
    import numpy as np
    from PIL import Image
    model = load_esrgan()
    img = np.array(pil_image.convert('RGB')).astype(np.float32) / 255.0
    h, w, _ = img.shape
    scale = 4
    out = np.zeros((h * scale, w * scale, 3), dtype=np.float32)
    with torch.no_grad():
        for y in range(0, h, tile):
            for x in range(0, w, tile):
                y0, x0 = max(0, y - tile_pad), max(0, x - tile_pad)
                y1, x1 = min(h, y + tile + tile_pad), min(w, x + tile + tile_pad)
                patch = torch.from_numpy(img[y0:y1, x0:x1].transpose(2, 0, 1)[None]).cuda()
                res = model(patch)[0].float().cpu().numpy().transpose(1, 2, 0)
                py0 = (y - y0) * scale
                px0 = (x - x0) * scale
                ph = min(tile, h - y) * scale
                pw = min(tile, w - x) * scale
                out[y * scale:y * scale + ph, x * scale:x * scale + pw] = res[py0:py0 + ph, px0:px0 + pw]
    out = np.clip(out * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(out)

In [ ]:
# 8) 煙霧測試 1：對話（最快，先試呢個）
try:
    print('=== 測試 Qwen3-8B ===')
    print(chat_reply('用一句廣東話介紹你自己。', max_tokens=128))
    print('PASS: Qwen3-8B')
except Exception as e:
    print('FAIL: Qwen3-8B -', repr(e))

In [ ]:
# 9) 煙霧測試 2：放大（整張細圖再放大，驗證流程）
try:
    from PIL import Image, ImageDraw
    img = Image.new('RGB', (64, 64), 'white')
    d = ImageDraw.Draw(img)
    d.ellipse([8, 8, 56, 56], fill='red')
    up = upscale_image(img)
    print('放大前:', img.size, '-> 放大後:', up.size)
    up.save('/kaggle/working/test_upscale.png')
    print('PASS: Real-ESRGAN，已存 /kaggle/working/test_upscale.png')
except Exception as e:
    print('FAIL: Real-ESRGAN -', repr(e))

In [ ]:
# 10) 煙霧測試 3：出圖（細張、10 步，求快，只驗證跑得郁）
try:
    img = make_image('a red apple on a wooden table, photo', width=512, height=512, steps=10, seed=42)
    print('出圖尺寸:', img.size)
    img.save('/kaggle/working/test_image.png')
    print('PASS: Qwen-Image-2.1，已存 /kaggle/working/test_image.png')
except Exception as e:
    print('FAIL: Qwen-Image-2.1 -', repr(e))

In [ ]:
# 11) 煙霧測試 4：短片（最慢：17 格、480p、20 步，只驗證全流程跑得郁）
try:
    path = make_video('a cat sitting on a windowsill, sunny day', width=832, height=480, num_frames=17, steps=20)
    print('PASS: Wan 2.2，短片已存:', path)
except Exception as e:
    print('FAIL: Wan 2.2 -', repr(e))

In [ ]:
# 12) 開 Gradio 四合一介面（最後一格；跑起之後會出公開連結，唔好停呢格）
import gradio as gr

def tab_image(prompt, width, height, steps, seed):
    if not prompt or not prompt.strip():
        return None, '請輸入 prompt'
    try:
        img = make_image(prompt, int(width), int(height), int(steps), int(seed))
        return img, '完成'
    except Exception as e:
        return None, f'出圖失敗：{e}'

def tab_video(prompt, image, num_frames, steps):
    if not prompt or not prompt.strip():
        return None, '請輸入 prompt'
    try:
        path = make_video(prompt, image=image, width=832, height=480,
                          num_frames=int(num_frames), steps=int(steps))
        return path, f'完成：{path}（480p，{int(num_frames)} 格，約 {int(num_frames) // 16} 秒）'
    except Exception as e:
        return None, f'出片失敗：{e}'

def tab_chat(message, history):
    try:
        return chat_reply(message, max_tokens=512)
    except Exception as e:
        return f'對話失敗：{e}'

def tab_upscale(image):
    if image is None:
        return None, '請上載圖片'
    try:
        up = upscale_image(image)
        return up, f'完成：{image.size} -> {up.size}（4x）'
    except Exception as e:
        return None, f'放大失敗：{e}'

with gr.Blocks(title='免費 GPU 四合一') as demo:
    gr.Markdown('# 免費 GPU 四合一：出圖・短片・對話・放大\nKaggle T4 x2 免費 GPU。第一次用某個 tab 嗰陣要等佢下載＋載入模型；短片最慢，請耐心等。')
    with gr.Tab('出圖'):
        p = gr.Textbox(label='Prompt（英文效果最好）', placeholder='a red apple on a wooden table, photo', lines=2)
        with gr.Row():
            w = gr.Dropdown(['512', '768', '1024'], value='1024', label='闊')
            h = gr.Dropdown(['512', '768', '1024'], value='1024', label='高')
            st = gr.Slider(10, 40, value=25, step=1, label='Steps（愈多愈細緻，愈慢）')
            sd = gr.Number(value=0, label='Seed（0 = 隨機）')
        btn = gr.Button('出圖', variant='primary')
        out = gr.Image(label='成品')
        msg = gr.Textbox(label='狀態', interactive=False)
        btn.click(tab_image, inputs=[p, w, h, st, sd], outputs=[out, msg])
    with gr.Tab('短片'):
        p2 = gr.Textbox(label='Prompt（英文效果最好）', placeholder='a cat sitting on a windowsill, sunny day', lines=2)
        im2 = gr.Image(label='參考圖（唔填 = 文字出片；填咗 = 圖片變片）', type='pil')
        with gr.Row():
            fr = gr.Dropdown(['17', '33', '49'], value='33', label='格數（17 格約 1 秒，33 格約 2 秒）')
            st2 = gr.Slider(15, 40, value=25, step=1, label='Steps')
        btn2 = gr.Button('出片（480p，T4 上要幾分鐘）', variant='primary')
        out2 = gr.Video(label='成品')
        msg2 = gr.Textbox(label='狀態', interactive=False)
        btn2.click(tab_video, inputs=[p2, im2, fr, st2], outputs=[out2, msg2])
    with gr.Tab('對話'):
        gr.ChatInterface(fn=tab_chat, title='Qwen3-8B（本地跑，唔使上網問人）')
    with gr.Tab('放大 4x'):
        im4 = gr.Image(label='上載圖片', type='pil')
        btn4 = gr.Button('放大 4 倍', variant='primary')
        out4 = gr.Image(label='放大後')
        msg4 = gr.Textbox(label='狀態', interactive=False)
        btn4.click(tab_upscale, inputs=[im4], outputs=[out4, msg4])
    with gr.Row():
        gr.Button('釋放全部模型（慳顯存）').click(unload_all, outputs=[])

demo.queue(max_size=5).launch(share=True, show_error=True)
print('Gradio 已開，上面有公開連結。')

## 用後須知

- 呢個 session 最長 12 小時，唔用嗰陣 Kaggle 會喺閒置 20 分鐘後斷線，Gradio 連結亦會隨之失效；要再用就重新開 notebook 跑過。
- 每週 GPU 額度約 30 小時，唔好長開唔用。
- 出短片係最慢嘅一環：T4 上 480p、2 秒片都要幾分鐘；想快啲可以將格數減到 17、steps 減到 20。
- 出圖想快啲：之後可以換 Turbo 版 transformer（unsloth/Qwen-Image-2.1-Turbo-FP8，8 步就出到圖）。
- Notebook 檔案本身可以喺 Kaggle 嗰度 Save，權重唔會跟住存，下次開要重新下載。

## 改版記錄

- v4（2026-11-09）：對話改用 bitsandbytes 4-bit（Qwen/Qwen3-8B 全精度＋NF4 量化），唔用 AWQ——gptqmodel 喺呢個環境冇預製 wheel，要由源碼 build CUDA，v3 嗰轉 build 咗成兩個鐘都未完；autoawq 一齊唔裝。安裝格而家全部係預製 wheel，唔使再等源碼 build。
- v3（2026-11-09）：版本全部釘死——diffusers 0.41.0＋transformers 5.17.0（Qwen 官方文件指定嘅配對，0.41.0 係第一個有 Qwen-Image 2.1 嘅正式版）；唔裝 torchao（最新版會將 torch 升到 2.15，搞爛 gptqmodel 同 diffusers）；放大改用 notebook 自帶 RRDBNet 程式碼，唔使裝 basicsr／realesrgan（嗰兩個喺 Python 3.13 裝唔到）；安裝後印埋 torch 版本，升咗會即刻見到。
- v2（2026-11-09）：安裝分開幾行獨立 pip，邊行失敗都唔影響其他；basicsr 用 `--no-build-isolation`（佢喺 Kaggle build wheel 成日失敗，拖垮成個安裝，令 gptqmodel 裝唔到）；煙霧測試同 Gradio 各 tab 加 try/except，個別失敗唔會停低成個 run；安裝後逐個 import 驗證，唔再靠一句 print扮完成。

（完）